In [42]:
%pip install ipywidgets


  Using cached ipywidgets-8.1.8-py3-none-any.whl.metadata (2.4 kB)
  Using cached widgetsnbextension-4.0.15-py3-none-any.whl.metadata (1.6 kB)
  Using cached jupyterlab_widgets-3.0.16-py3-none-any.whl.metadata (20 kB)
Using cached ipywidgets-8.1.8-py3-none-any.whl (139 kB)
Using cached jupyterlab_widgets-3.0.16-py3-none-any.whl (914 kB)
Using cached widgetsnbextension-4.0.15-py3-none-any.whl (2.2 MB)

   -------------------------- ------------- 2/3 [ipywidgets]
   -------------------------- ------------- 2/3 [ipywidgets]
   -------------------------- ------------- 2/3 [ipywidgets]
   -------------------------- ------------- 2/3 [ipywidgets]
   ---------------------------------------- 3/3 [ipywidgets]

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
target_path = "C:\\Users\\wongb\\Bridge-ML\\Bridge-ML-LLM-Embedding-Architecture\\part1 clean\\original data\\targets_10Sep.csv"
NBIfeatures_path = "C:\\Users\\wongb\\Bridge-ML\\Bridge-ML-LLM-Embedding-Architecture\\data\\NBIfull.csv"

In [2]:
import pandas as pd

# === Load Data ===
targets = pd.read_csv(target_path)
targets_all = pd.read_csv(target_path)
print(f"✅ Starting Number of Bridges: {len(targets):,}")
print(f"   Shape: {targets.shape}")

# === Track filtering impact ===
row_counts = {"Initial": len(targets)}

# 1️⃣ Support Type removals
support_type_removals = {
    "Not in BEISt": "Excluded due to missing in BEISt",
    "Not Bridge": "Not an actual bridge",
    "Single Span": "Single-span bridges excluded",
    "Wall": "Wall support type removed",
    "Tapered Wall": "Tapered wall support type removed",
    "CIP Piles": "CIP pile supports removed",
    "Timber piles": "Timber piles removed",
    "Timber Piles": "Timber piles removed (alt casing)",
    "Composite Piles": "Composite piles removed",
    "PC Piles": "PC piles removed",
    "Piles": "Generic piles removed",
    "H-Piles": "H-piles removed",
    "Tapered Col": "Tapered columns removed",
}

print("\n🔍 Support Type Removals:")
for val, reason in support_type_removals.items():
    count = (targets["Support Type"] == val).sum()
    if count > 0:
        print(f"  - {reason}: {count:,} rows")
        targets = targets[targets["Support Type"] != val].copy()

row_counts["After Support Type Filter"] = len(targets)
print(f"\n✅ After Support Type Filter: {len(targets):,} rows (removed {row_counts['Initial'] - len(targets):,})")

# 2️⃣ Structure ID filter
structure_id_exclude = ["Not in BEISt"]
count_before = len(targets)
for value in structure_id_exclude:
    count = (targets["Structure ID"] == value).sum()
    if count > 0:
        print(f"\n🔍 Structure ID Filter: Removing {count:,} rows where Structure ID == '{value}'")
        targets = targets[targets["Structure ID"] != value].copy()

row_counts["After Structure ID Filter"] = len(targets)
print(f"✅ After Structure ID Filter: {len(targets):,} rows (removed {count_before - len(targets):,})")

# 3️⃣ Feature Intersected filter
feature_intersected_exclude = ["Not Bridge"]
count_before = len(targets)
for value in feature_intersected_exclude:
    count = (targets["Feature Intersected"] == value).sum()
    if count > 0:
        print(f"\n🔍 Feature Intersected Filter: Removing {count:,} rows where Feature Intersected == '{value}'")
        targets = targets[targets["Feature Intersected"] != value].copy()

row_counts["After Feature Intersected Filter"] = len(targets)
print(f"✅ After Feature Intersected Filter: {len(targets):,} rows (removed {count_before - len(targets):,})")

# === Summary ===
print("\n" + "="*60)
print("📊 FILTERING SUMMARY:")
print("="*60)
for step, count in row_counts.items():
    print(f"{step:.<40} {count:>6,} rows")

print("\n" + "="*60)
print(f"🎯 Final Shape: {targets.shape}")
print(f"📉 Total Removed: {row_counts['Initial'] - len(targets):,} rows ({(row_counts['Initial'] - len(targets)) / row_counts['Initial'] * 100:.1f}%)")
print("="*60)

print("\n🔢 Remaining Support Types:")
print(targets["Support Type"].value_counts())



✅ Starting Number of Bridges: 850
   Shape: (850, 11)

🔍 Support Type Removals:
  - Excluded due to missing in BEISt: 8 rows
  - Not an actual bridge: 27 rows
  - Single-span bridges excluded: 115 rows
  - Wall support type removed: 40 rows
  - CIP pile supports removed: 4 rows
  - Timber piles removed: 29 rows
  - Composite piles removed: 1 rows
  - PC piles removed: 76 rows
  - H-piles removed: 2 rows
  - Tapered columns removed: 45 rows

✅ After Support Type Filter: 503 rows (removed 347)

🔍 Structure ID Filter: Removing 16 rows where Structure ID == 'Not in BEISt'
✅ After Structure ID Filter: 487 rows (removed 16)
✅ After Feature Intersected Filter: 487 rows (removed 0)

📊 FILTERING SUMMARY:
Initial.................................    850 rows
After Support Type Filter...............    503 rows
After Structure ID Filter...............    487 rows
After Feature Intersected Filter........    487 rows

🎯 Final Shape: (487, 11)
📉 Total Removed: 363 rows (42.7%)

🔢 Remaining Support Ty

In [3]:
# --- Rename columns for consistency ---
rename_map = {
    'Structure ID': 'STRUCTURE_ID',
    'Spacing/Pitch (in)': 'SPACING_PITCH_IN',
    'No. of Columns/Piles per Bent': 'COLUMNS_PILES_PER_BENT',
    'B_long (in.)': 'B_LONG_IN',
    'B_trans (in.)': 'B_TRANS_IN',
    'Clear Height (ft)': 'CLEAR_HEIGHT_FT',
    'Support Type': 'SUPPORT_TYPE',
    'Feature Intersected': 'OBJECT_INTERSECTED',
    'Long Reinf Ratio, %': 'LRR',
    'Transverse Reinforcement Ratio': 'TRR',
    'Database Ref #': 'DATABASE_REF'
}

missing_cols = [col for col in rename_map if col not in targets.columns]
valid_renames = {k: v for k, v in rename_map.items() if k in targets.columns}
targets = targets.rename(columns=valid_renames)
targets_all = targets_all.rename(columns=valid_renames)

print(f"✅ Renamed {len(valid_renames)} columns for consistency.")
if missing_cols:
    print(f"⚠️ Skipped missing columns: {missing_cols}")

✅ Renamed 11 columns for consistency.


In [4]:
import numpy as np
import pandas as pd

# --- Engineered / Calculated Features ---

print("\n🧩 Adding engineered features...")

# === 1️⃣ Columns/Piles 0/1 ===
# Ensure numeric type first — coerce non-numerics to NaN
targets["COLUMNS_PILES_PER_BENT"] = pd.to_numeric(
    targets["COLUMNS_PILES_PER_BENT"], errors="coerce"
)

# Convert dimension columns to numeric
targets["B_LONG_IN"] = pd.to_numeric(targets["B_LONG_IN"], errors="coerce")
targets["B_TRANS_IN"] = pd.to_numeric(targets["B_TRANS_IN"], errors="coerce")
targets["CLEAR_HEIGHT_FT"] = pd.to_numeric(targets["CLEAR_HEIGHT_FT"], errors="coerce")
targets["LRR"] = pd.to_numeric(targets["LRR"], errors="coerce")
targets["TRR"] = pd.to_numeric(targets["TRR"], errors="coerce")

# 0 if exactly one pile/column per bent, 1 if more than one, NaN if missing.
targets["COLUMNS_PILES_0_1"] = np.select(
    [
        targets["COLUMNS_PILES_PER_BENT"].notna() & (targets["COLUMNS_PILES_PER_BENT"] == 1),
        targets["COLUMNS_PILES_PER_BENT"].notna() & (targets["COLUMNS_PILES_PER_BENT"] > 1),
    ],
    [0, 1],
    default=np.nan,
)
# Convert to nullable integer using pd.array for compatibility
targets["COLUMNS_PILES_0_1"] = pd.array(targets["COLUMNS_PILES_0_1"], dtype=pd.Int64Dtype())

# === 2️⃣ L/H Ratios ===
# Multiply by 12 to convert ft to in before ratio.
targets["L_H_LONG"] = (targets["CLEAR_HEIGHT_FT"] * 12) / targets["B_LONG_IN"]
targets["L_H_TRANS"] = (targets["CLEAR_HEIGHT_FT"] * 12) / targets["B_TRANS_IN"]

# Minimum slenderness ratio
targets["L_H_MIN"] = targets[["L_H_LONG", "L_H_TRANS"]].min(axis=1)

# === 3️⃣ Material / Design Constants ===
f_prime_c = 6000  # psi
f_y = 60000       # psi
f_y_t = 60000     # psi
eta = 0.5
pi_4 = np.pi / 4
C_0 = 1.6
C_1 = 1.3

sqrt_fc_fy = np.sqrt(f_prime_c) / f_y
fyt_fy = f_y_t / f_y

# === 4️⃣ Z_KNOWN (from pile/column presence) ===
targets["Z_KNOWN"] = targets["COLUMNS_PILES_0_1"].map({0: 1, 1: 2})

# === 5️⃣ Capacity / Demand Ratios ===
# The following calculations originate from the "Helper Tables" shear formulas in the old database.
# Some were commented out historically — kept as optional formulas for reference.

# Optional / inactive metrics (preserved for clarity)
# targets["CD_LONG"] = (
#     (C_0 * sqrt_fc_fy + 0.4 * targets["TRR"] * fyt_fy * targets["L_H_LONG"])
#     / (C_1 * 1 * pi_4 * eta * targets["LRR"])
# )
#
# targets["CD_TRANS"] = (
#     (C_0 * sqrt_fc_fy + 0.4 * targets["TRR"] * fyt_fy * targets["L_H_TRANS"])
#     / (C_1 * targets["Z_KNOWN"] * pi_4 * eta * targets["LRR"])
# )
#
# targets["CD_MIN"] = targets[["CD_LONG", "CD_TRANS"]].min(axis=1)

# Active formula used in previous version:
targets["CD_MIN"] = (
    (C_0 * sqrt_fc_fy + 0.5 * targets["TRR"] * fyt_fy) * targets["L_H_MIN"]
) / (C_1 * targets["Z_KNOWN"] * pi_4 * eta * targets["LRR"])

# Handle divide-by-zero or missing LRR safely
targets.loc[targets["LRR"] == 0, "CD_MIN"] = np.nan

# === 6️⃣ Critical Truth 0/1 ===
# 1 if CD_MIN < 0.85, 0 otherwise, NaN if missing.
targets["CRITICAL_TRUTH_0_1"] = np.where(
    targets["CD_MIN"].notna(),
    np.where(targets["CD_MIN"] < 0.85, 1, 0),
    np.nan,
)
# Convert to nullable integer using pd.array for compatibility
targets["CRITICAL_TRUTH_0_1"] = pd.array(targets["CRITICAL_TRUTH_0_1"], dtype=pd.Int64Dtype())

print("✅ Engineered features added successfully.")



🧩 Adding engineered features...
✅ Engineered features added successfully.


In [5]:
targets_pre =  targets
targets


,DATABASE_REF,STRUCTURE_ID,OBJECT_INTERSECTED,SUPPORT_TYPE,COLUMNS_PILES_PER_BENT,B_LONG_IN,B_TRANS_IN,CLEAR_HEIGHT_FT,LRR,SPACING_PITCH_IN,TRR,COLUMNS_PILES_0_1,L_H_LONG,L_H_TRANS,L_H_MIN,Z_KNOWN,CD_MIN,CRITICAL_TRUTH_0_1
0,1,0019133A,Overpass,RectCol-Spiral,4.0,54.0,54.0,19.20,0.0155,6,0.0077,1,4.266667,4.266667,4.266667,2.0,1.594858,0
1,2,0019133D,Overpass,RectCol-Spiral,2.0,48.0,48.0,24.60,0.0140,4,0.0096,1,6.150000,6.150000,6.150000,2.0,2.953873,0
6,7,0005582A,Overpass,CircCol,4.0,36.0,36.0,34.40,0.0123,12,0.0020,1,11.466667,11.466667,11.466667,2.0,2.799066,0
7,8,0005523C,Overpass,CircCol,1.0,60.0,60.0,18.00,0.0099,12,0.0011,0,3.600000,3.600000,3.600000,1.0,1.863090,0
8,9,0005651A,Overpass,CircCol,1.0,72.0,72.0,25.20,0.0092,12,0.0010,0,4.200000,4.200000,4.200000,1.0,2.294276,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
842,843,0014567C,Terrain,CircCol,2.0,48.0,48.0,8.10,0.0172,3.5,0.0077,1,2.025000,2.025000,2.025000,2.0,0.682121,1
846,847,0014354A,Underpass,CircCol,4.0,36.0,36.0,25.31,0.0118,3.5,0.0067,1,8.436667,8.436667,8.436667,2.0,3.792290,0
847,848,0006068B,Underpass,CircCol,5.0,36.0,36.0,28.40,NaN,Missing,NaN,1,9.466667,9.466667,9.466667,2.0,NaN,<NA>
848,849,0017936A,Underpass,RectCol-Spiral,1.0,48.0,72.0,18.41,0.0199,4,NaN,0,4.602500,3.068333,3.068333,1.0,NaN,<NA>


In [6]:
if "CD_MIN" not in targets.columns:
    raise KeyError("targets must contain 'CD_MIN' before filtering")

rows_before = len(targets)
targets = targets[targets["CD_MIN"].notna()].copy()
rows_after = len(targets)

print(f"✅ Removed rows with NaN CD_MIN: {rows_before - rows_after:,}")
print(f"📌 Remaining rows in targets: {rows_after:,}")
targets

✅ Removed rows with NaN CD_MIN: 110
📌 Remaining rows in targets: 377


,DATABASE_REF,STRUCTURE_ID,OBJECT_INTERSECTED,SUPPORT_TYPE,COLUMNS_PILES_PER_BENT,B_LONG_IN,B_TRANS_IN,CLEAR_HEIGHT_FT,LRR,SPACING_PITCH_IN,TRR,COLUMNS_PILES_0_1,L_H_LONG,L_H_TRANS,L_H_MIN,Z_KNOWN,CD_MIN,CRITICAL_TRUTH_0_1
0,1,0019133A,Overpass,RectCol-Spiral,4.0,54.0,54.0,19.20,0.0155,6,0.0077,1,4.266667,4.266667,4.266667,2.0,1.594858,0
1,2,0019133D,Overpass,RectCol-Spiral,2.0,48.0,48.0,24.60,0.0140,4,0.0096,1,6.150000,6.150000,6.150000,2.0,2.953873,0
6,7,0005582A,Overpass,CircCol,4.0,36.0,36.0,34.40,0.0123,12,0.0020,1,11.466667,11.466667,11.466667,2.0,2.799066,0
7,8,0005523C,Overpass,CircCol,1.0,60.0,60.0,18.00,0.0099,12,0.0011,0,3.600000,3.600000,3.600000,1.0,1.863090,0
8,9,0005651A,Overpass,CircCol,1.0,72.0,72.0,25.20,0.0092,12,0.0010,0,4.200000,4.200000,4.200000,1.0,2.294276,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
837,838,0014354B,Water,RectCol,2.0,42.0,57.0,46.60,0.0149,3.5,0.0217,1,13.314286,9.810526,9.810526,2.0,8.328889,0
840,841,0016611B,Water,CircCol,2.0,60.0,60.0,47.30,0.0184,3.5,0.0087,1,9.460000,9.460000,9.460000,2.0,3.230552,0
841,842,0016611A,Underpass,CircCol,2.0,60.0,60.0,38.70,0.0127,3.5,0.0087,1,7.740000,7.740000,7.740000,2.0,3.829488,0
842,843,0014567C,Terrain,CircCol,2.0,48.0,48.0,8.10,0.0172,3.5,0.0077,1,2.025000,2.025000,2.025000,2.0,0.682121,1


In [7]:
import re
import pandas as pd

# Path to the master/total dataset (update as needed)
master_path = r"C:\Users\wongb\Bridge-ML\Bridge-ML-LLM-Embedding-Architecture\part1 clean\master data\segments\sst_ready\total_dataset.csv"

master_df = pd.read_csv(master_path)

if "unified_id" not in master_df.columns:
    raise KeyError("master_df must contain a 'unified_id' column")
if "STRUCTURE_ID" not in targets.columns:
    raise KeyError("targets must contain a 'STRUCTURE_ID' column")

target_ids = (
    targets["STRUCTURE_ID"]
    .dropna()
    .astype(str)
    .str.strip()
    .loc[lambda s: s != ""]
    .unique()
    .tolist()
)

if not target_ids:
    raise ValueError("No valid STRUCTURE_ID values found in targets")

# Keep rows where unified_id text contains any target STRUCTURE_ID
pattern = "|".join(re.escape(structure_id) for structure_id in target_ids)
master_df = master_df.copy()
master_df["_unified_id_text"] = master_df["unified_id"].fillna("").astype(str)

metadata_rows = master_df[
    master_df["_unified_id_text"].str.contains(pattern, regex=True, na=False)
].copy()

# Optional: recover which STRUCTURE_ID matched first in the unified_id string
metadata_rows["STRUCTURE_ID"] = metadata_rows["_unified_id_text"].str.extract(
    f"({'|'.join(re.escape(structure_id) for structure_id in target_ids)})", expand=False
)

metadata_rows = metadata_rows.drop(columns=["_unified_id_text"], errors="ignore")

print(f"✅ Matched metadata rows: {len(metadata_rows):,}")
print(f"   Unique STRUCTURE_ID matched: {metadata_rows['STRUCTURE_ID'].nunique():,}")

metadata_rows.head()

✅ Matched metadata rows: 370
   Unique STRUCTURE_ID matched: 370


,unified_id,PGA,S1,SDS,FPGA,MACRO_AGE_MIN,MIN_VERT_CLR_010,NAV_VERT_CLR_MT_039,NAV_HORR_CLR_MT_040,HORR_CLR_MT_047,...,MEMBRANE_TYPE_108B,DECK_PROTECTION_108C,NATIONAL_NETWORK_110,PIER_PROTECTION_111,SCOUR_CRITICAL_113,LOWEST_RATING,NFHL_FLD_ZONE,NFHL_SFHA,tokens,STRUCTURE_ID
74,"('000000QC', '(47.41211111, -122.2215)')",0.496719,0.056699,0.579544,-0.614846,-0.310921,-999.0,-999.0,-999.0,1.578138,...,1,1,1,-999.0,9,7,2.0,2.0,"[101, 22125, 3131, 2171, 1024, 2035, 2226, 573...",000000QC
147,"('0001406A', '(48.06611111, -124.23525)')",1.521071,2.079430,0.912858,-0.614846,-0.310248,-999.0,-999.0,-999.0,-0.623754,...,3,1,2,-999.0,6,7,8.0,1.0,"[101, 22125, 3131, 2171, 1024, 17381, 11852, 1...",0001406A
159,"('0001576A', '(46.87949167, -123.271575)')",0.790664,0.967839,0.649307,-0.614846,-0.310921,-999.0,-999.0,-999.0,-0.747224,...,3,1,2,-999.0,4,7,1.0,2.0,"[101, 22125, 3131, 2171, 1024, 2035, 2226, 573...",0001576A
161,"('0001576B', '(46.84793333, -123.2485556)')",0.692682,0.894948,0.525284,-0.614846,-0.310921,-999.0,-999.0,-999.0,-0.747224,...,3,1,2,-999.0,11,6,8.0,1.0,"[101, 22125, 3131, 2171, 1024, 2035, 2226, 573...",0001576B
171,"('0001679A', '(46.97824167, -123.7913889)')",1.672497,2.061208,1.354692,-0.614846,-0.162544,-999.0,-999.0,-999.0,-0.273920,...,1,1,2,-999.0,11,8,8.0,1.0,"[101, 22125, 3131, 2171, 1024, 26926, 1011, 20...",0001679A


In [8]:
import time
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from catboost import CatBoostRegressor

# --- CD_MIN Imputation Pipeline ---

df = targets_pre.copy()

if "CD_MIN" not in df.columns:
    raise KeyError("targets_pre must contain 'CD_MIN'")
if "STRUCTURE_ID" not in df.columns:
    raise KeyError("targets_pre must contain 'STRUCTURE_ID' as key")

key_col = "STRUCTURE_ID"
pipeline_start = time.perf_counter()

known_mask = df["CD_MIN"].notna()
missing_mask = ~known_mask

n_known = int(known_mask.sum())
n_missing = int(missing_mask.sum())

print(f"Known CD_MIN rows:   {n_known:,}")
print(f"Missing CD_MIN rows: {n_missing:,}")

# FIX #5: Fail fast before any heavy processing
if n_missing > 0 and n_known < 1:
    raise ValueError("Cannot impute: no known CD_MIN rows available to train on")

if n_missing == 0:
    # Keep traceability columns consistent even when nothing is imputed
    df["CD_MIN_original"] = df["CD_MIN"]
    df["CD_MIN_filled"] = df["CD_MIN"]
    df["is_cd_min_imputed"] = 0
    print("✅ No missing CD_MIN values to impute.")
else:
    # Explicitly exclude target/leakage columns (including leftovers from prior runs)
    exclude_cols = {
        "CD_MIN",
        "CRITICAL_TRUTH_0_1",
        "DATABASE_REF",
        key_col,
        "CD_MIN_original",
        "CD_MIN_filled",
        "is_cd_min_imputed",
    }
    feature_cols = [c for c in df.columns if c not in exclude_cols]

    X = df[feature_cols].copy()
    y = df["CD_MIN"].astype(float)

    # FIX #3: Convert bool columns to int (0/1) instead of string categories
    bool_cols = X.select_dtypes(include=["bool"]).columns.tolist()
    for col in bool_cols:
        X[col] = X[col].astype(int)

    cat_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()
    for col in cat_cols:
        X[col] = X[col].astype("string").fillna("__MISSING__")

    X_known = X.loc[known_mask].copy()
    y_known = y.loc[known_mask].copy()
    X_missing = X.loc[missing_mask].copy()

    # FIX #6: Sanity-check that X_missing is non-empty
    assert len(X_missing) > 0, "missing_mask produced zero rows — logic error"

    cat_feature_idx = [X_known.columns.get_loc(col) for col in cat_cols]

    base_params = {
        "loss_function": "RMSE",
        "eval_metric": "RMSE",
        "random_seed": 42,
        "verbose": False,
    }
    best_params = {
        "iterations": 700,
        "depth": 6,
        "learning_rate": 0.05,
        "l2_leaf_reg": 3,
    }

    # FIX #1: Evaluate on a held-out validation split for a realistic RMSE estimate,
    # then retrain on all known rows before imputing.
    if n_known >= 5:
        X_tr, X_val, y_tr, y_val = train_test_split(
            X_known, y_known, test_size=0.2, random_state=42
        )
        eval_model = CatBoostRegressor(**base_params, **best_params)
        eval_model.fit(X_tr, y_tr, cat_features=cat_feature_idx)

        val_preds = eval_model.predict(X_val)
        mse = mean_squared_error(y_val, val_preds)
        rmse = float(np.sqrt(mse))
        y_range = float(y_known.max() - y_known.min())
        nrmse = np.nan if y_range == 0 else (rmse / y_range)

        print(f"\n--- Validation metrics (held-out 20%) ---")
        print(f"Val MSE:  {mse:.6f}")
        print(f"Val RMSE: {rmse:.6f}")
        if np.isnan(nrmse):
            print("Val NRMSE: NaN (CD_MIN range is 0)")
        else:
            print(f"Val NRMSE: {nrmse:.6f} ({nrmse * 100:.2f}%)")
        print(f"-----------------------------------------\n")
    else:
        print(
            f"⚠️  Only {n_known} known rows — skipping train/val split, "
            "metrics will be optimistic (trained & evaluated on same data)."
        )
        eval_model = CatBoostRegressor(**base_params, **best_params)
        eval_model.fit(X_known, y_known, cat_features=cat_feature_idx)
        train_preds = eval_model.predict(X_known)
        mse = mean_squared_error(y_known, train_preds)
        rmse = float(np.sqrt(mse))
        y_range = float(y_known.max() - y_known.min())
        nrmse = np.nan if y_range == 0 else (rmse / y_range)
        print(f"Train MSE:  {mse:.6f}  ⚠️  optimistic")
        print(f"Train RMSE: {rmse:.6f}  ⚠️  optimistic")
        if not np.isnan(nrmse):
            print(f"Train NRMSE: {nrmse:.6f} ({nrmse * 100:.2f}%)  ⚠️  optimistic")

    # Retrain on ALL known rows before imputing missing values
    final_model = CatBoostRegressor(**base_params, **best_params)
    final_model.fit(X_known, y_known, cat_features=cat_feature_idx)

    imputed_values = final_model.predict(X_missing)

    # Create/refresh traceability columns
    df["CD_MIN_original"] = df["CD_MIN"]
    df["CD_MIN_filled"] = df["CD_MIN"]
    df["is_cd_min_imputed"] = 0

    df.loc[missing_mask, "CD_MIN_filled"] = imputed_values
    df.loc[missing_mask, "is_cd_min_imputed"] = 1

    imputed_keys = df.loc[missing_mask, key_col]
    print(f"Imputed keys (sample): {imputed_keys.head().tolist()}")
    print("Params used:", best_params)

# Intentionally reassigns outer-scope targets_pre with the imputed copy
targets_pre = df

print("✅ Imputation complete.")
print("Added columns: CD_MIN_original, CD_MIN_filled, is_cd_min_imputed")
print(targets_pre[[key_col, "CD_MIN", "CD_MIN_filled", "is_cd_min_imputed"]].head())
print(f"Total pipeline time: {time.perf_counter() - pipeline_start:.2f}s")

Known CD_MIN rows:   377
Missing CD_MIN rows: 110

--- Validation metrics (held-out 20%) ---
Val MSE:  0.092273
Val RMSE: 0.303764
Val NRMSE: 0.034952 (3.50%)
-----------------------------------------

Imputed keys (sample): ['0007326B', '0018607C', '0006792A', '0006792B', '0006480A']
Params used: {'iterations': 700, 'depth': 6, 'learning_rate': 0.05, 'l2_leaf_reg': 3}
✅ Imputation complete.
Added columns: CD_MIN_original, CD_MIN_filled, is_cd_min_imputed
  STRUCTURE_ID    CD_MIN  CD_MIN_filled  is_cd_min_imputed
0     0019133A  1.594858       1.594858                  0
1     0019133D  2.953873       2.953873                  0
6     0005582A  2.799066       2.799066                  0
7     0005523C  1.863090       1.863090                  0
8     0005651A  2.294276       2.294276                  0
Total pipeline time: 34.60s


In [9]:
targets_pre

,DATABASE_REF,STRUCTURE_ID,OBJECT_INTERSECTED,SUPPORT_TYPE,COLUMNS_PILES_PER_BENT,B_LONG_IN,B_TRANS_IN,CLEAR_HEIGHT_FT,LRR,SPACING_PITCH_IN,...,COLUMNS_PILES_0_1,L_H_LONG,L_H_TRANS,L_H_MIN,Z_KNOWN,CD_MIN,CRITICAL_TRUTH_0_1,CD_MIN_original,CD_MIN_filled,is_cd_min_imputed
0,1,0019133A,Overpass,RectCol-Spiral,4.0,54.0,54.0,19.20,0.0155,6,...,1,4.266667,4.266667,4.266667,2.0,1.594858,0,1.594858,1.594858,0
1,2,0019133D,Overpass,RectCol-Spiral,2.0,48.0,48.0,24.60,0.0140,4,...,1,6.150000,6.150000,6.150000,2.0,2.953873,0,2.953873,2.953873,0
6,7,0005582A,Overpass,CircCol,4.0,36.0,36.0,34.40,0.0123,12,...,1,11.466667,11.466667,11.466667,2.0,2.799066,0,2.799066,2.799066,0
7,8,0005523C,Overpass,CircCol,1.0,60.0,60.0,18.00,0.0099,12,...,0,3.600000,3.600000,3.600000,1.0,1.863090,0,1.863090,1.863090,0
8,9,0005651A,Overpass,CircCol,1.0,72.0,72.0,25.20,0.0092,12,...,0,4.200000,4.200000,4.200000,1.0,2.294276,0,2.294276,2.294276,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
842,843,0014567C,Terrain,CircCol,2.0,48.0,48.0,8.10,0.0172,3.5,...,1,2.025000,2.025000,2.025000,2.0,0.682121,1,0.682121,0.682121,0
846,847,0014354A,Underpass,CircCol,4.0,36.0,36.0,25.31,0.0118,3.5,...,1,8.436667,8.436667,8.436667,2.0,3.792290,0,3.792290,3.792290,0
847,848,0006068B,Underpass,CircCol,5.0,36.0,36.0,28.40,NaN,Missing,...,1,9.466667,9.466667,9.466667,2.0,NaN,<NA>,NaN,3.956474,1
848,849,0017936A,Underpass,RectCol-Spiral,1.0,48.0,72.0,18.41,0.0199,4,...,0,4.602500,3.068333,3.068333,1.0,NaN,<NA>,NaN,1.005708,1


In [10]:
from pathlib import Path

output_dir = Path(r"C:\Users\wongb\Bridge-ML\Bridge-ML-LLM-Embedding-Architecture\part1 clean\master data\segments\sst_ready")
labeled_output_path = output_dir / "labeled_dataset.csv"
targets_pre.to_csv(labeled_output_path, index=False)

print(f"Saved labeled dataset to: {labeled_output_path}")
print(f"Shape: {targets_pre.shape}")

Saved labeled dataset to: C:\Users\wongb\Bridge-ML\Bridge-ML-LLM-Embedding-Architecture\part1 clean\master data\segments\sst_ready\labeled_dataset.csv
Shape: (487, 21)
